In [0]:
#import required module 

from pyspark.sql.functions import *
from pyspark.sql import *
from pyspark.sql.types import *

In [0]:
#define variable 
Storage_name = "stytdev001"

#configure authentication for storage
spark.conf.set(
    f"fs.azure.account.auth.type.{Storage_name}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{Storage_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-id"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{Storage_name}.dfs.core.windows.net",dbutils.secrets.get(scope="app_yt_scope", key="app-yt-client-secrets"))


spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{Storage_name}.dfs.core.windows.net",
    "https://login.microsoftonline.com/65583477-d432-469c-b4af-ab046853916f/oauth2/v2.0/token",
)


In [0]:
#define widgets for table name and storage path

dbutils.widgets.text('TABLE_NAME','')
dbutils.widgets.text('STORAGE_PATH','')

TABLE_NAME = dbutils.widgets.get('TABLE_NAME')
STORAGE_PATH = dbutils.widgets.get('STORAGE_PATH')

In [0]:
#Define source and sink path

source_path = f"abfss://global@{Storage_name}.dfs.core.windows.net/gold/"


# STORAGE_FOLDER_NAME = TABLE_NAME + '_output/'

full_source_path = source_path + STORAGE_PATH + '/' + "dim" + TABLE_NAME + '/'
full_sink_path = "dim_" + TABLE_NAME


print(full_source_path)
print(full_sink_path)

abfss://global@stytdev001.dfs.core.windows.net/gold/fare_rules_output/dimfare_rules/
dim_fare_rules


In [0]:
#read Data
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(full_source_path)


In [0]:
sfOptions = {
    "sfURL": "PNKPEWR-BW37274.snowflakecomputing.com",
    "sfUser": "AFZAL1991",
    "sfPassword": dbutils.secrets.get(
        scope="app_yt_scope",
        key="snowflake-password"
    ),
    "sfDatabase": "YELLOW_TAXI_DB",
    "sfSchema": "YELLOW_TAXI_SCHEMA",
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "ACCOUNTADMIN"
}

SNOWFLAKE_SOURCE_NAME = "net.snowflake.spark.snowflake"

In [0]:
#Write data into snowflake yellow_taxi_db database

df.write \
    .format("net.snowflake.spark.snowflake") \
    .options(**sfOptions) \
    .option("dbtable", full_sink_path) \
    .mode("overwrite") \
    .save()